In [6]:
#r "C:\Users\user\Desktop\Lukyanov\practice2026\task14\bin\Release\net10.0\task14.dll"
#r "nuget: ScottPlot, 5.0.21"

Installed Packages ScottPlot, 5.0.21

Loading extensions from `C:\Users\user\.nuget\packages\skiasharp\2.88.7\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

In [7]:
using System;
using System.Diagnostics;
using System.IO;
using System.Collections.Generic;
using System.Linq;
using task14;

double a = -100.0;
double b = 100.0;
Func<double, double> targetFunction = Math.Sin;
double targetPrecision = 1e-4;
double exactValue = 0.0;

double[] stepsToTest = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };
int[] threadCounts = { 1, 2, 4, 6, 8, 12, 16 };
int runCount = 5;

double chosenStep = 0;
int optimalThreads = 0;
double bestMultiTime = 0;
List<double> bestAverageTimes = new List<double>();
double finalPercentageDiff = 0;

foreach (var step in stepsToTest)
{
    double res = DefiniteIntegral.SolveSingleThreaded(a, b, targetFunction, step);
    double error = Math.Abs(res - exactValue);
    Console.WriteLine($"\n=== Тестирование шага: {step:E1} ===");
    Console.WriteLine($"Погрешность: {error:E2} (требуемая: {targetPrecision:E1})");
    
    if (error > targetPrecision)
    {
        Console.WriteLine($"Шаг {step:E1} не обеспечивает требуемую точность, пропускаем...");
        continue;
    }
    
    List<double> averageTimes = new List<double>();
    
    foreach (int threads in threadCounts)
    {
        List<double> intermediateTimes = new List<double>();
        
        for (int r = 0; r < runCount; r++)
        {
            Stopwatch sw = Stopwatch.StartNew();
            
            if (threads == 1)
            {
                DefiniteIntegral.SolveSingleThreaded(a, b, targetFunction, step);
            }
            else
            {
                DefiniteIntegral.Solve(a, b, targetFunction, step, threads);
            }
            
            sw.Stop();
            intermediateTimes.Add(sw.Elapsed.TotalMilliseconds);
        }
        
        double avgTime = intermediateTimes.Average();
        averageTimes.Add(avgTime);
        Console.WriteLine($"Потоков: {threads}. Среднее время: {avgTime:F2} мс");
    }
    
    double singleTime = averageTimes[0];
    double multiTime = averageTimes.Skip(1).Min();
    double percentageDiff = ((singleTime - multiTime) / singleTime) * 100;
    Console.WriteLine($"КПД для шага {step:E1}: {percentageDiff:F2}%");
    
    if (percentageDiff > 15.0)
    {
        chosenStep = step;
        optimalThreads = threadCounts[averageTimes.IndexOf(multiTime)];
        bestMultiTime = multiTime;
        bestAverageTimes = averageTimes;
        finalPercentageDiff = percentageDiff;
        Console.WriteLine($"✓ Найден оптимальный шаг: {step:E1}");
        break;
    }
    else
    {
        Console.WriteLine($"Шаг {step:E1} не подходит: КПД = {percentageDiff:F2}% <= 15%");
    }
}

if (chosenStep == 0)
{
    Console.WriteLine("\nОШИБКА: Не найден шаг с КПД > 15% и требуемой точностью!");
    return;
}

Console.WriteLine($"\nИТОГОВЫЕ РЕЗУЛЬТАТЫ");
Console.WriteLine($"ВЫБРАН ШАГ: {chosenStep:E1}");
Console.WriteLine($"ОПТИМАЛЬНОЕ ЧИСЛО ПОТОКОВ: {optimalThreads} (Время: {bestMultiTime:F2} мс)");

Console.WriteLine("\nСравнение с чистой однопоточной реализацией");
double avgSingleTime = bestAverageTimes[0];

Console.WriteLine($"Однопоточная версия. Среднее время: {avgSingleTime:F2} мс");
Console.WriteLine($"Оптимальная многопоточная. Среднее время: {bestMultiTime:F2} мс");
Console.WriteLine($"Разница в производительности: {finalPercentageDiff:F2}%");

if (finalPercentageDiff < 15.0)
{
    Console.WriteLine("Ошибка. Ускорение меньше 15%.");
}
else
{
    Console.WriteLine("Успешно. Ускорение больше 15%.");
}

var plt = new ScottPlot.Plot();

double[] plotX = bestAverageTimes.ToArray();
double[] plotY = threadCounts.Select(t => (double)t).ToArray();

var scatter = plt.Add.Scatter(plotX, plotY);
scatter.LineWidth = 3;
scatter.MarkerSize = 10;
scatter.Color = ScottPlot.Color.FromHex("#1e3a8a");

plt.Title($"Зависимость производительности от количества потоков (шаг {chosenStep:E1})");
plt.XLabel("Время вычисления (мс)");
plt.YLabel("Количество потоков");

string timestamp = DateTime.Now.Ticks.ToString();
string fileName = $"threads_performance_{chosenStep:E1}_{timestamp}.png";

if (File.Exists(fileName))
    File.Delete(fileName);

plt.SavePng(fileName, 700, 500);
Console.WriteLine($"График сохранен: {fileName}");

string reportPath = "performance_report.txt";
using (StreamWriter writer = new StreamWriter(reportPath, false, System.Text.Encoding.UTF8))
{
    writer.WriteLine("Лукьянов Денис Валерьевич Фит-252");
    writer.WriteLine();
    writer.WriteLine($"1. Выбранный размер шага: {chosenStep:E1}");
    writer.WriteLine($"   Данный шаг выполняет {(int)((b - a) / chosenStep):N0} итераций интегрирования на отрезке [{a}, {b}].");
    writer.WriteLine();
    writer.WriteLine($"2. Оптимальное количество потоков: {optimalThreads}");
    writer.WriteLine();
    writer.WriteLine($"3. Скорость работы и сравнение реализаций:");
    writer.WriteLine($"   - Среднее время работы однопоточной версии: {avgSingleTime:F2} мс");
    writer.WriteLine($"   - Среднее время работы многопоточной версии ({optimalThreads} пот.): {bestMultiTime:F2} мс");
    writer.WriteLine($"   - Разница в производительности: {finalPercentageDiff:F2}%");
}

if (File.Exists(fileName))
{
    display(HTML($"<img src='{fileName}?t={timestamp}' width='700'/>"));
}


=== Тестирование шага: 1.0E-001 ===
Погрешность: 1.58E-014 (требуемая: 1.0E-004)
Потоков: 1. Среднее время: 0.17 мс
Потоков: 2. Среднее время: 0.74 мс
Потоков: 4. Среднее время: 0.69 мс
Потоков: 6. Среднее время: 1.13 мс
Потоков: 8. Среднее время: 1.12 мс
Потоков: 12. Среднее время: 1.68 мс
Потоков: 16. Среднее время: 2.93 мс
КПД для шага 1.0E-001: -312.82%
Шаг 1.0E-001 не подходит: КПД = -312.82% <= 15%

=== Тестирование шага: 1.0E-002 ===
Погрешность: 4.46E-016 (требуемая: 1.0E-004)
Потоков: 1. Среднее время: 0.42 мс
Потоков: 2. Среднее время: 0.63 мс
Потоков: 4. Среднее время: 0.80 мс
Потоков: 6. Среднее время: 1.01 мс
Потоков: 8. Среднее время: 1.41 мс
Потоков: 12. Среднее время: 2.09 мс
Потоков: 16. Среднее время: 2.55 мс
КПД для шага 1.0E-002: -47.58%
Шаг 1.0E-002 не подходит: КПД = -47.58% <= 15%

=== Тестирование шага: 1.0E-003 ===
Погрешность: 6.92E-015 (требуемая: 1.0E-004)
Потоков: 1. Среднее время: 6.03 мс
Потоков: 2. Среднее время: 2.88 мс
Потоков: 4. Среднее время: 2.99 

<null>